# Noise and Random Signals

`rand`, `rand_interp`, pink/brown noise, and the smoothing generators -- the raw material for percussion, breath and textures.

Source: [CLM generators -- rand](https://ccrma.stanford.edu/software/snd/snd/sndclm.html)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import IPython.display as ipd

import pysndlib.clm as clm
import pysndlib.generators as gens

SR = 44100

def spectrum(x, n=8192):
    rl = np.copy(x[:n].astype(float)); im = np.zeros(n)
    clm.mus_fft(rl, im, n, 1)
    clm.mus_rectangular2polar(rl, im)
    return np.abs(rl[:n//2])

## rand vs rand_interp

`rand` picks a new random value every 1/frequency seconds; `rand_interp` glides between picks. Both are band-limited by construction. At low "frequencies" the difference is dramatic:

In [ ]:
r1 = clm.make_rand(20.0)
r2 = clm.make_rand_interp(20.0)
plt.figure(figsize=(14, 3))
plt.plot([clm.rand(r1) for _ in range(2000)], lw=0.8, label='rand')
plt.plot([clm.rand_interp(r2) for _ in range(2000)], lw=0.8, label='rand_interp')
plt.legend(); plt.show()

## Pink and brown noise

The `pysndlib.generators` module adds noise colors. Pink noise falls off at -3 dB/octave (equal energy per octave), brown noise at -6 dB/octave:

In [ ]:
pink = gens.make_pink_noise(12)
brown = gens.make_brown_noise(10.0)
pk = np.array([pink.next() for _ in range(SR)])
br = np.array([brown.next() for _ in range(SR)])
pk = pk / np.abs(pk).max(); br = br / np.abs(br).max()

n = 8192
f = np.arange(n//2) * SR / n
plt.figure(figsize=(14, 3))
plt.loglog(f[1:], spectrum(pk)[1:] + 1e-6, lw=0.7, label='pink')
plt.loglog(f[1:], spectrum(br)[1:] + 1e-6, lw=0.7, label='brown')
plt.legend(); plt.xlabel('Hz'); plt.title('noise colors'); plt.show()

In [ ]:
ipd.Audio(pk, rate=SR)

In [ ]:
ipd.Audio(br, rate=SR)

## Smoothing generators

`moving_average`, `moving_max` and `moving_norm` window their input -- useful for envelopes and "wind" textures:

In [ ]:
noise = np.random.default_rng(3).uniform(-1, 1, SR)
ma = clm.make_moving_average(64)
out = np.array([clm.moving_average(ma, float(x)) for x in noise])
plt.figure(figsize=(14, 3))
plt.plot(noise[:3000], alpha=0.3, label='input')
plt.plot(out[:3000], lw=0.8, label='moving_average(64)')
plt.legend(); plt.show()

## A noise instrument

Filtered noise bursts through a resonant filter are percussion -- here is a snare-ish drum:

In [ ]:
with clm.Sound('snare.wav', channels=1, srate=SR):
    for hit in range(8):
        beg = hit * 0.25
        start = clm.seconds2samples(beg)
        noise_gen = clm.make_rand(8000.0)
        tone = clm.make_oscil(190.0)
        res = clm.make_one_pole(0.25, -0.75)
        amp = clm.make_env([0, 1, 20, 0.4, 100, 0], scaler=0.5, duration=0.18)
        for i in range(start, start + clm.seconds2samples(0.18)):
            v = clm.env(amp) * (0.7 * clm.rand(noise_gen) + 0.3 * clm.oscil(tone))
            clm.outa(i, clm.one_pole(res, v))

In [ ]:
ipd.Audio('snare.wav', rate=SR)

## Summary

- `rand` vs `rand_interp` is a smoothness choice; colors come from `pysndlib.generators`
- Smoothing generators window any input signal
- Noise + resonance = percussion

Next: [delays and physical modeling](11_Delays_Physical.ipynb)